# 장미 잎 탐색 + 상태 판정 (Colab)

- **1단계 잎 탐색:** 기존 세그멘테이션 모델 `leaf_seg_v8r2.pt`
- **2단계 상태 분류:** 새 분류 모델 `leaf_cls_yolov8m.pt` (YOLOv8m-cls: black_spot, downy_mildew, healthy, insects_infected)

**실행 순서**
1. 메뉴에서 **런타임 > 런타임 유형 변경 > T4 GPU**를 선택합니다.
2. `rose_leaf_colab_package.zip`을 Google Drive의 `내 드라이브` 바로 아래에 올려 둡니다. 없으면 2번 셀에서 직접 업로드하라고 안내합니다.
3. 위에서부터 셀을 차례대로 실행합니다.

## 1. GPU 확인

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 실행되어 느립니다. 런타임 유형을 T4 GPU로 바꾸세요)")

## 2. 패키지 가져오기 및 설치

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive, files

USE_DRIVE = True  # Drive를 쓰지 않으려면 False
ZIP_NAME = "rose_leaf_colab_package.zip"
WORK = Path("/content/rose_leaf")

zip_path = None
if USE_DRIVE:
    drive.mount("/content/drive")
    p = Path("/content/drive/MyDrive") / ZIP_NAME
    if p.exists():
        zip_path = p
    else:
        print("Drive에서 찾지 못했습니다:", p)
if zip_path is None:
    print(ZIP_NAME, "파일을 선택해 업로드하세요")
    zip_path = Path("/content") / next(iter(files.upload()))

shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(WORK)
print("압축 해제 완료:", sorted(x.name for x in WORK.iterdir()))

In [ ]:
!pip install -q -r /content/rose_leaf/requirements.txt

## 3. 사진 올리기

사진(jpg, png, bmp, webp)을 여러 장 선택할 수 있습니다. 아무것도 선택하지 않고 취소하면 포함된 샘플 사진으로 실행합니다.

In [ ]:
PHOTOS = Path("/content/photos")
shutil.rmtree(PHOTOS, ignore_errors=True)
PHOTOS.mkdir()

try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
for name in uploaded:
    shutil.move(f"/content/{name}", PHOTOS / name)
if not uploaded:
    shutil.copy(WORK / "samples" / "fix_eafda190.jpg", PHOTOS)
    print("샘플 사진으로 실행합니다")
print(sorted(x.name for x in PHOTOS.iterdir()))

## 4. 실행

In [ ]:
RESULTS = Path("/content/results")
shutil.rmtree(RESULTS, ignore_errors=True)
!cd /content/rose_leaf && python run.py /content/photos --out /content/results

## 5. 결과 보기

| 상태 | 의미 |
|---|---|
| FRESH | 건강 |
| ANOMALY:black_spot / downy_mildew / insects_infected | 이상 (클래스 표시) |
| UNKNOWN | 이상 의심, 관찰 대상 |
| HOLD_SHAPE | 잎이 합쳐졌거나 사진 가장자리에 잘림, 판정 안 함 |

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(RESULTS / "summary.csv"))
for f in sorted(RESULTS.glob("*_vis.jpg")):
    print(f.name)
    display(Image(filename=str(f), width=1000))

이상 또는 의심으로 판정된 잎 목록입니다.

In [ ]:
rows = []
for f in sorted(RESULTS.glob("*.json")):
    d = json.loads(f.read_text())
    for l in d["leaves"]:
        if l["state"] in ("ANOMALY", "UNKNOWN"):
            rows.append(dict(photo=d["photo"], id=l["id"], state=l["state"], cls=l["anomaly_class"], box=l["box"], **l["probs"]))
display(pd.DataFrame(rows) if rows else "이상 또는 의심 잎 없음")

## 6. 결과 내려받기

In [ ]:
shutil.make_archive("/content/results", "zip", RESULTS)
files.download("/content/results.zip")